In [ ]:
%%capture
!pip install llama-index llama-index-embeddings-openai qdrant-client llama-index-vector-stores-qdrant llama-index llama-index-llms-openai

In [ ]:
import os
from getpass import getpass
import nest_asyncio
from dotenv import load_dotenv
from llama_index.llms.openai import OpenAI
from llama_index.core.settings import Settings

nest_asyncio.apply()

load_dotenv()

In [13]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY') or getpass("Enter your OpenAI API key: ")
TAVILY_API_KEY=os.environ.get('TAVILY_API_KEY') or getpass("Enter your TAVILY API key: ")

In [ ]:
Settings.llm = OpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)

In [ ]:
%pip install tavily-python

In [14]:
from tavily import AsyncTavilyClient
from llama_index.core.workflow import Context


async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient(api_key=TAVILY_API_KEY)
    return str(await client.search(query))


async def record_notes(ctx: Context, notes: str, notes_title: str) -> str:
    """Useful for recording notes on a given topic. Your input should be notes with a title to save the notes under."""
    current_state = await ctx.get("state")
    if "research_notes" not in current_state:
        current_state["research_notes"] = {}
    current_state["research_notes"][notes_title] = notes
    await ctx.set("state", current_state)
    return "Notes recorded."


async def write_report(ctx: Context, report_content: str) -> str:
    """Useful for writing a report on a given topic. Your input should be a markdown formatted report."""
    current_state = await ctx.get("state")
    current_state["report_content"] = report_content
    await ctx.set("state", current_state)
    return "Report written."


async def review_report(ctx: Context, review: str) -> str:
    """Useful for reviewing a report and providing feedback. Your input should be a review of the report."""
    current_state = await ctx.get("state")
    current_state["review"] = review
    await ctx.set("state", current_state)
    return "Report reviewed."

In [15]:
from llama_index.core.agent.workflow import FunctionAgent, ReActAgent

research_agent = FunctionAgent(
    name="ResearchAgent",
    description="Useful for searching the web for information on a given topic and recording notes on the topic.",
    system_prompt=(
        "You are the ResearchAgent that can search the web for information on a given topic and record notes on the topic. "
        "Once notes are recorded and you are satisfied, you should hand off control to the WriteAgent to write a report on the topic. "
        "You should have at least some notes on a topic before handing off control to the WriteAgent."
    ),
    llm=Settings.llm,
    tools=[search_web, record_notes],
    can_handoff_to=["WriteAgent"],
)

write_agent = FunctionAgent(
    name="WriteAgent",
    description="Useful for writing a report on a given topic.",
    system_prompt=(
        "You are the WriteAgent that can write a report on a given topic. "
        "Your report should be in a markdown format. The content should be grounded in the research notes. "
        "Once the report is written, you should get feedback at least once from the ReviewAgent."
    ),
    llm=Settings.llm,
    tools=[write_report],
    can_handoff_to=["ReviewAgent", "ResearchAgent"],
)

review_agent = FunctionAgent(
    name="ReviewAgent",
    description="Useful for reviewing a report and providing feedback.",
    system_prompt=(
        "You are the ReviewAgent that can review the write report and provide feedback. "
        "Your review should either approve the current report or request changes for the WriteAgent to implement. "
        "If you have feedback that requires changes, you should hand off control to the WriteAgent to implement the changes after submitting the review."
    ),
    llm=Settings.llm,
    tools=[review_report],
    can_handoff_to=["WriteAgent"],
)

In [16]:
from llama_index.core.agent.workflow import AgentWorkflow

agent_workflow = AgentWorkflow(
    agents=[research_agent, write_agent, review_agent],
    root_agent=research_agent.name,
    initial_state={
        "research_notes": {},
        "report_content": "Not written yet.",
        "review": "Review required.",
    },
)

In [19]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

handler = agent_workflow.run(
    user_msg=(
        "Write me a report on computer system validation. "
        "Briefly describe the main steps and challenges"
        "Give recommendations on implementation of CSV"
    )
)

current_agent = None
current_tool_calls = ""
async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")

    # if isinstance(event, AgentStream):
    #     if event.delta:
    #         print(event.delta, end="", flush=True)
    # elif isinstance(event, AgentInput):
    #     print("📥 Input:", event.input)
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")


🤖 Agent: ResearchAgent

🛠️  Planning to use tools: ['search_web']
🔨 Calling Tool: search_web
  With arguments: {'query': 'computer system validation main steps and challenges'}
🔧 Tool Result (search_web):
  Arguments: {'query': 'computer system validation main steps and challenges'}
  Output: {'query': 'computer system validation main steps and challenges', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'What Is Computer System Validation, and How Do I Do It Right?', 'url': 'https://www.csolsinc.com/resources/what-is-computer-system-validation-and-how-do-i-do-it-right', 'content': 'Computer System Validation Challenges. Validation of computer systems can involve challenges, including the risk of system failure, restrictive company policies, and increasingly stringent regulatory requirements. Another significant issue is when users need to balance the risk vs. cost equation after risk categories are defined.', 'score': 0.875031, 'raw_content': None}, {

In [20]:
~state = await handler.ctx.get("state")
print(state["report_content"])

# Computer System Validation (CSV)

## Introduction

Computer System Validation (CSV) is a critical process in regulated industries such as pharmaceuticals, healthcare, and biotechnology. It ensures that computer systems meet specific regulatory requirements and perform as intended, producing accurate and compliant results. This report outlines the main steps and challenges of CSV and provides recommendations for its effective implementation.

## Main Steps in Computer System Validation

1. **Planning**: This initial phase involves defining the scope of validation, identifying the systems to be validated, and developing a validation plan that outlines the approach, resources, and timelines.

2. **Specification**: Detailed specifications are developed to define the system requirements, including functional, operational, and performance criteria. This step ensures that the system is designed to meet user needs and regulatory standards.

3. **Testing**: Rigorous testing is conducted to ve

# Computer System Validation (CSV)

## Introduction

Computer System Validation (CSV) is a critical process in regulated industries such as pharmaceuticals, healthcare, and biotechnology. It ensures that computer systems meet specific regulatory requirements and perform as intended, producing accurate and compliant results. This report outlines the main steps and challenges of CSV and provides recommendations for its effective implementation.

## Main Steps in Computer System Validation

1. **Planning**: This initial phase involves defining the scope of validation, identifying the systems to be validated, and developing a validation plan that outlines the approach, resources, and timelines.

2. **Specification**: Detailed specifications are developed to define the system requirements, including functional, operational, and performance criteria. This step ensures that the system is designed to meet user needs and regulatory standards.

3. **Testing**: Rigorous testing is conducted to verify that the system meets the specified requirements. This includes unit testing, integration testing, and user acceptance testing to ensure the system functions correctly in all scenarios.

4. **Reporting**: Comprehensive documentation is produced to provide evidence of the validation process. This includes test results, deviations, and corrective actions, ensuring transparency and traceability.

## Challenges in Computer System Validation

- **System Complexity**: As systems become more complex, ensuring all components are validated can be challenging.
- **Regulatory Compliance**: Keeping up with evolving regulations and ensuring compliance can be resource-intensive.
- **Cost and Resource Allocation**: Balancing the cost of validation with the need for thorough testing and documentation can be difficult.
- **Risk Management**: Identifying and mitigating risks associated with system failures is crucial.

## Recommendations for Implementing CSV

- **Regulatory Compliance**: Ensure adherence to relevant regulations, such as those from the FDA, to maintain compliance and system reliability.
- **Effective Planning and Project Management**: Develop a clear validation plan and manage resources efficiently to meet validation goals.
- **Engage External Consultants**: Consider hiring external experts to provide guidance and training, especially in complex validation scenarios.
- **Continuous Improvement**: Regularly review and update validation practices to incorporate new technologies and methodologies.
- **Effective Communication and Training**: Ensure all stakeholders are informed and trained on CSV processes to enhance understanding and execution.
- **Data Integrity**: Maintain robust data management practices to ensure the integrity and traceability of information.

## Conclusion

Computer System Validation is essential for ensuring that systems in regulated industries operate reliably and produce accurate results. By following structured validation processes and addressing common challenges, organizations can achieve compliance and enhance system performance. Implementing the recommended practices will help streamline CSV efforts and ensure successful outcomes.